# 06 - Machine Learning models

**Goal:** train several classical models on the same data and compare them.

| Model | Why | Uses scaled data |
|---|---|---|
| Logistic Regression | Simple linear baseline | Yes |
| Decision Tree | Simple non-linear baseline, easy to interpret | No |
| Random Forest | Many trees voting (bagging), robust | No |
| XGBoost | Gradient boosting, usually the best on tabular data | No |
| LightGBM | Faster gradient boosting | No |
| Soft voting (bonus) | Average of Random Forest + XGBoost + LightGBM | No |
| LightGBM tuned (bonus) | LightGBM with hyperparameters found by Optuna | No |

**Class imbalance:**
1. The training set is **capped** at `TRAIN_CAP` rows per class (big classes are randomly reduced, which also speeds up training)
2. **Class weights**: errors on rare classes cost more during training
3. The **test set is not changed**: it keeps the real distribution

Each model saves its predictions and metrics to `results/`, and the trained model to `models/`.

In [1]:
# Imports
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib
from google.colab import drive

# Mount Drive
drive.mount("/content/drive")

# Paths
BASE = Path("/content/drive/MyDrive/Projects/CSE-CIC-IDS2018")
DATA_DIR = BASE / "data" / "processed"
READY_DIR = BASE / "data" / "model_ready"
MODELS_DIR = BASE / "models"
RESULTS_DIR = BASE / "results"
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
SEED = 42
pd.set_option("display.max_columns", 100)

Mounted at /content/drive


## 0. Load the data (notebook 05)

In [2]:
X_train = pd.read_parquet(READY_DIR / "X_train.parquet")
X_test = pd.read_parquet(READY_DIR / "X_test.parquet")
y_train = np.load(READY_DIR / "y_train.npy")
y_test = np.load(READY_DIR / "y_test.npy")
scaler = joblib.load(READY_DIR / "scaler.joblib")
encoder = joblib.load(READY_DIR / "label_encoder.joblib")

class_names = list(encoder.classes_)
BENIGN = class_names.index("Benign")
print("Train:", X_train.shape, "| Test:", X_test.shape, "| Classes:", len(class_names))

Train: (2372881, 49) | Test: (593221, 49) | Classes: 11


## 1. Training set: cap the big classes
Keep at most `TRAIN_CAP` random rows per class. Small classes are kept entirely.

In [3]:
TRAIN_CAP = 150_000        # lower it (e.g. 50_000) for a faster first run

train = X_train.copy()
train["y"] = y_train
train = train.sample(frac=1, random_state=SEED).groupby("y").head(TRAIN_CAP)

X_tr = train.drop(columns="y")
y_tr = train["y"].to_numpy()
del train

pd.DataFrame({
    "train (capped)": pd.Series(np.array(class_names)[y_tr]).value_counts(),
    "test": pd.Series(np.array(class_names)[y_test]).value_counts(),
})

,train (capped),test
Benign,150000,328660
Bot,115628,28907
DDOS attack-HOIC,150000,39772
DDOS attack-LOIC-UDP,1384,346
DDoS attacks-LOIC-HTTP,150000,115003
DoS attacks-GoldenEye,33114,8278
DoS attacks-Hulk,116159,29040
DoS attacks-Slowloris,7770,1942
Infilteration,89178,22295
SSH-Bruteforce,75233,18808


### Class weights and scaled data
- `sample_weight` gives each class the same total weight (used by XGBoost)
- Logistic Regression needs scaled features, so we prepare scaled copies with the scaler from notebook 05

In [4]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weight = compute_sample_weight("balanced", y_tr)

X_tr_scaled = scaler.transform(X_tr)
X_test_scaled = scaler.transform(X_test)

## 2. Evaluation function
Computes the metrics on the test set and saves them with the predictions.

| Metric | Meaning |
|---|---|
| **macro-F1** (main metric) | Average F1 over all classes, each class counts equally |
| accuracy | Share of correct predictions (misleading with imbalance) |
| balanced accuracy | Average recall over the classes |
| **detection rate** | Share of real attacks flagged as an attack (any type) |
| **false alarm rate** | Share of Benign flows wrongly flagged as an attack |

In [5]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report)

def evaluate(name, y_pred, y_proba=None, train_time=None):
    is_attack = y_test != BENIGN
    pred_attack = y_pred != BENIGN

    metrics = {
        "model": name,
        "macro_f1": f1_score(y_test, y_pred, average="macro"),
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
        "detection_rate": (is_attack & pred_attack).sum() / is_attack.sum(),
        "false_alarm_rate": (~is_attack & pred_attack).sum() / (~is_attack).sum(),
        "train_time_s": train_time,
    }

    np.save(RESULTS_DIR / f"{name}_y_pred.npy", y_pred.astype(np.int16))
    if y_proba is not None:
        np.save(RESULTS_DIR / f"{name}_y_proba.npy", y_proba.astype(np.float32))
    json.dump(metrics, open(RESULTS_DIR / f"{name}_metrics.json", "w"), indent=2)

    print(f"{name}: macro-F1 = {metrics['macro_f1']:.4f} | accuracy = {metrics['accuracy']:.4f} | "
          f"detection = {metrics['detection_rate']:.4f} | false alarms = {metrics['false_alarm_rate']:.4f}")
    print(classification_report(y_test, y_pred, target_names=class_names, digits=4, zero_division=0))
    return metrics

### Training helper
Trains a model, measures the training time, predicts on the test set, saves the model and evaluates it.
Returns the predicted probabilities (used later by the soft voting ensemble).

In [6]:
def run_model(name, model, X_fit, X_eval, sample_weight=None):
    start = time.time()
    model.fit(X_fit, y_tr, sample_weight=sample_weight)
    train_time = time.time() - start

    y_proba = model.predict_proba(X_eval)
    y_pred = y_proba.argmax(axis=1)

    joblib.dump(model, MODELS_DIR / f"{name}.joblib", compress=3)
    evaluate(name, y_pred, y_proba, train_time)
    return y_proba

## 3. Logistic Regression (baseline)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, class_weight="balanced")
proba_lr = run_model("LogisticRegression", lr, X_tr_scaled, X_test_scaled)

LogisticRegression: macro-F1 = 0.7946 | accuracy = 0.7717 | detection = 0.9717 | false alarms = 0.3863
                        precision    recall  f1-score   support

                Benign     0.9643    0.6137    0.7500    328660
                   Bot     0.9765    0.9895    0.9830     28907
      DDOS attack-HOIC     0.9898    0.9998    0.9948     39772
  DDOS attack-LOIC-UDP     0.7061    1.0000    0.8278       346
DDoS attacks-LOIC-HTTP     0.9974    0.9976    0.9975    115003
 DoS attacks-GoldenEye     0.9872    0.9987    0.9929      8278
      DoS attacks-Hulk     0.9993    0.9990    0.9992     29040
 DoS attacks-Slowloris     0.9243    1.0000    0.9607      1942
         Infilteration     0.1077    0.6486    0.1848     22295
        SSH-Bruteforce     0.9992    0.9999    0.9995     18808
            Web-Attack     0.0259    0.9882    0.0504       170

              accuracy                         0.7717    593221
             macro avg     0.7889    0.9305    0.7946    593221

## 4. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(class_weight="balanced", min_samples_leaf=2, random_state=SEED)
proba_dt = run_model("DecisionTree", dt, X_tr, X_test)
print("Tree depth:", dt.get_depth(), "| leaves:", dt.get_n_leaves())

DecisionTree: macro-F1 = 0.8888 | accuracy = 0.8756 | detection = 0.9781 | false alarms = 0.2066
                        precision    recall  f1-score   support

                Benign     0.9783    0.7934    0.8762    328660
                   Bot     0.9992    0.9999    0.9996     28907
      DDOS attack-HOIC     0.9998    1.0000    0.9999     39772
  DDOS attack-LOIC-UDP     0.8354    0.9538    0.8907       346
DDoS attacks-LOIC-HTTP     0.9997    0.9994    0.9995    115003
 DoS attacks-GoldenEye     0.9992    0.9995    0.9993      8278
      DoS attacks-Hulk     0.9997    0.9999    0.9998     29040
 DoS attacks-Slowloris     0.9918    1.0000    0.9959      1942
         Infilteration     0.1962    0.7413    0.3103     22295
        SSH-Bruteforce     0.9998    1.0000    0.9999     18808
            Web-Attack     0.5824    0.8941    0.7053       170

              accuracy                         0.8756    593221
             macro avg     0.8711    0.9438    0.8888    593221
     

## 5. Random Forest
`max_depth=25` limits the size of the saved model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=25, min_samples_leaf=2,
                            class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
proba_rf = run_model("RandomForest", rf, X_tr, X_test)

RandomForest: macro-F1 = 0.8910 | accuracy = 0.8966 | detection = 0.9798 | false alarms = 0.1701
                        precision    recall  f1-score   support

                Benign     0.9808    0.8299    0.8991    328660
                   Bot     0.9993    1.0000    0.9997     28907
      DDOS attack-HOIC     0.9979    1.0000    0.9990     39772
  DDOS attack-LOIC-UDP     0.7818    0.9942    0.8753       346
DDoS attacks-LOIC-HTTP     0.9993    0.9991    0.9992    115003
 DoS attacks-GoldenEye     0.9995    0.9996    0.9996      8278
      DoS attacks-Hulk     0.9998    1.0000    0.9999     29040
 DoS attacks-Slowloris     0.9974    0.9990    0.9982      1942
         Infilteration     0.2339    0.7610    0.3578     22295
        SSH-Bruteforce     0.9997    1.0000    0.9998     18808
            Web-Attack     0.5324    0.9176    0.6739       170

              accuracy                         0.8966    593221
             macro avg     0.8656    0.9546    0.8910    593221
     

## 6. XGBoost
Class imbalance is handled with `sample_weight`. On a GPU runtime you can add `device="cuda"` to train faster.

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.8,
                    colsample_bytree=0.8, tree_method="hist", eval_metric="mlogloss",
                    n_jobs=-1, random_state=SEED)
proba_xgb = run_model("XGBoost", xgb, X_tr, X_test, sample_weight=sample_weight)

XGBoost: macro-F1 = 0.8822 | accuracy = 0.8682 | detection = 0.9830 | false alarms = 0.2240
                        precision    recall  f1-score   support

                Benign     0.9827    0.7760    0.8672    328660
                   Bot     0.9999    1.0000    0.9999     28907
      DDOS attack-HOIC     1.0000    1.0000    1.0000     39772
  DDOS attack-LOIC-UDP     0.8177    0.9855    0.8938       346
DDoS attacks-LOIC-HTTP     0.9998    0.9992    0.9995    115003
 DoS attacks-GoldenEye     0.9998    0.9996    0.9997      8278
      DoS attacks-Hulk     0.9999    1.0000    0.9999     29040
 DoS attacks-Slowloris     0.9985    0.9995    0.9990      1942
         Infilteration     0.1953    0.7989    0.3138     22295
        SSH-Bruteforce     0.9998    1.0000    0.9999     18808
            Web-Attack     0.4735    0.9471    0.6314       170

              accuracy                         0.8682    593221
             macro avg     0.8606    0.9551    0.8822    593221
          

## 7. LightGBM

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63, subsample=0.8,
                      subsample_freq=1, colsample_bytree=0.8, class_weight="balanced",
                      n_jobs=-1, random_state=SEED, verbose=-1)
proba_lgbm = run_model("LightGBM", lgbm, X_tr, X_test)

LightGBM: macro-F1 = 0.8574 | accuracy = 0.8752 | detection = 0.9845 | false alarms = 0.2126
                        precision    recall  f1-score   support

                Benign     0.9844    0.7874    0.8750    328660
                   Bot     0.9999    1.0000    0.9999     28907
      DDOS attack-HOIC     1.0000    1.0000    1.0000     39772
  DDOS attack-LOIC-UDP     0.8232    0.9827    0.8959       346
DDoS attacks-LOIC-HTTP     0.9999    0.9994    0.9996    115003
 DoS attacks-GoldenEye     0.9998    0.9998    0.9998      8278
      DoS attacks-Hulk     0.9999    1.0000    1.0000     29040
 DoS attacks-Slowloris     0.9985    1.0000    0.9992      1942
         Infilteration     0.2083    0.8165    0.3319     22295
        SSH-Bruteforce     0.9999    1.0000    1.0000     18808
            Web-Attack     0.1986    0.9706    0.3297       170

              accuracy                         0.8752    593221
             macro avg     0.8375    0.9597    0.8574    593221
         

## 8. (Bonus) Soft voting ensemble
Average the probabilities of **Random Forest + XGBoost + LightGBM**, then pick the class with the highest average.
- The three models make different errors, so averaging can cancel some of them.
- No retraining: we reuse the probabilities computed above.
- It is not guaranteed to win: if one model is much better, averaging can pull it down.

In [ ]:
proba_vote = (proba_rf + proba_xgb + proba_lgbm) / 3
y_vote = proba_vote.argmax(axis=1)

evaluate("SoftVoting", y_vote, proba_vote)

SoftVoting: macro-F1 = 0.8915 | accuracy = 0.8853 | detection = 0.9834 | false alarms = 0.1935
                        precision    recall  f1-score   support

                Benign     0.9837    0.8065    0.8863    328660
                   Bot     0.9999    1.0000    0.9999     28907
      DDOS attack-HOIC     1.0000    1.0000    1.0000     39772
  DDOS attack-LOIC-UDP     0.8132    0.9942    0.8947       346
DDoS attacks-LOIC-HTTP     0.9999    0.9993    0.9996    115003
 DoS attacks-GoldenEye     0.9996    0.9996    0.9996      8278
      DoS attacks-Hulk     0.9999    1.0000    0.9999     29040
 DoS attacks-Slowloris     0.9985    1.0000    0.9992      1942
         Infilteration     0.2202    0.8034    0.3457     22295
        SSH-Bruteforce     0.9998    1.0000    0.9999     18808
            Web-Attack     0.5273    0.9647    0.6819       170

              accuracy                         0.8853    593221
             macro avg     0.8675    0.9607    0.8915    593221
       

{'model': 'SoftVoting',
 'macro_f1': 0.8915303479752308,
 'accuracy': 0.8852535564317514,
 'balanced_accuracy': np.float64(0.9607035719215599),
 'weighted_f1': 0.912190270738415,
 'detection_rate': np.float64(0.9834178129051523),
 'false_alarm_rate': np.float64(0.19350088237083918),
 'train_time_s': None}

## 9. First comparison

In [ ]:
results = pd.DataFrame([json.load(open(f)) for f in RESULTS_DIR.glob("*_metrics.json")])
results = results.set_index("model").sort_values("macro_f1", ascending=False)
results.round(4)

,macro_f1,accuracy,balanced_accuracy,weighted_f1,detection_rate,false_alarm_rate,train_time_s
model,,,,,,,
SoftVoting,0.8915,0.8853,0.9607,0.9122,0.9834,0.1935,NaN
RandomForest,0.8910,0.8966,0.9546,0.9195,0.9798,0.1701,243.6939
DecisionTree,0.8888,0.8756,0.9438,0.9052,0.9781,0.2066,35.8393
XGBoost,0.8822,0.8682,0.9551,0.9004,0.9830,0.2240,539.4589
LightGBM,0.8574,0.8752,0.9597,0.9053,0.9845,0.2126,454.1188
LogisticRegression,0.7946,0.7717,0.9305,0.8286,0.9717,0.3863,294.3823


## 10. (Bonus) Hyperparameter tuning with Optuna
Optuna tries different hyperparameters for **LightGBM** and keeps the best (Bayesian optimization).
- Tuning uses a **validation set taken from the training data**: the test set is never used for tuning.
- Objective: macro-F1. It runs on a smaller sample (`TUNE_CAP` rows per class) to stay fast.
- The best parameters are then used to retrain on the full training set.

Set `RUN_OPTUNA = False` to skip.

In [ ]:
!pip install -q optuna

In [ ]:
RUN_OPTUNA = True
N_TRIALS = 30
TUNE_CAP = 30_000

if RUN_OPTUNA:
    import optuna
    from sklearn.model_selection import train_test_split
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    tune = X_tr.copy()
    tune["y"] = y_tr
    tune = tune.groupby("y").head(TUNE_CAP)
    X_a, X_val, y_a, y_val = train_test_split(tune.drop(columns="y"), tune["y"],
                                              test_size=0.25, stratify=tune["y"], random_state=SEED)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=100),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 255, log=True),
            "max_depth": trial.suggest_int("max_depth", 4, 16),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        }
        model = LGBMClassifier(**params, subsample_freq=1, class_weight="balanced",
                               n_jobs=-1, random_state=SEED, verbose=-1)
        model.fit(X_a, y_a)
        return f1_score(y_val, model.predict(X_val), average="macro")

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    print("Best validation macro-F1:", round(study.best_value, 4))
    print("Best parameters:", study.best_params)

  0%|          | 0/30 [00:00<?, ?it/s]

Best validation macro-F1: 0.9542
Best parameters: {'n_estimators': 900, 'learning_rate': 0.016615134094680642, 'num_leaves': 49, 'max_depth': 15, 'min_child_samples': 7, 'subsample': 0.6558415864770006, 'colsample_bytree': 0.7395595255054788}


In [ ]:
if RUN_OPTUNA:
    scores = [t.value for t in study.trials]
    plt.figure(figsize=(8, 3.5))
    plt.plot(scores, "o", alpha=0.5, label="trial")
    plt.plot(np.maximum.accumulate(scores), "-", label="best so far")
    plt.xlabel("Trial")
    plt.ylabel("Validation macro-F1")
    plt.title("Optuna optimization history")
    plt.legend()
    plt.show()

    lgbm_tuned = LGBMClassifier(**study.best_params, subsample_freq=1, class_weight="balanced",
                                n_jobs=-1, random_state=SEED, verbose=-1)
    run_model("LightGBM_tuned", lgbm_tuned, X_tr, X_test)

In [8]:
from lightgbm import LGBMClassifier

def show_progress(env):
    if (env.iteration + 1) % 50 == 0:
        print(f"{env.iteration + 1}/{env.end_iteration} trees — {time.time() - start:.0f} s", flush=True)

lgbm_tuned = LGBMClassifier(**best_params, subsample_freq=1, class_weight="balanced",
                            n_jobs=-1, random_state=SEED, verbose=-1)

start = time.time()
lgbm_tuned.fit(X_tr, y_tr, callbacks=[show_progress])
train_time = time.time() - start

y_proba = lgbm_tuned.predict_proba(X_test)
joblib.dump(lgbm_tuned, MODELS_DIR / "LightGBM_tuned.joblib", compress=3)
evaluate("LightGBM_tuned", y_proba.argmax(axis=1), y_proba, train_time)

50/900 trees — 88 s
100/900 trees — 166 s
150/900 trees — 242 s
200/900 trees — 317 s
250/900 trees — 389 s
300/900 trees — 459 s
350/900 trees — 525 s
400/900 trees — 595 s
450/900 trees — 666 s
500/900 trees — 739 s
550/900 trees — 811 s
600/900 trees — 885 s
650/900 trees — 959 s
700/900 trees — 1035 s
750/900 trees — 1116 s
800/900 trees — 1202 s
850/900 trees — 1286 s
900/900 trees — 1363 s
LightGBM_tuned: macro-F1 = 0.8566 | accuracy = 0.8730 | detection = 0.9837 | false alarms = 0.2158
                        precision    recall  f1-score   support

                Benign     0.9835    0.7842    0.8726    328660
                   Bot     0.9999    1.0000    0.9999     28907
      DDOS attack-HOIC     1.0000    1.0000    1.0000     39772
  DDOS attack-LOIC-UDP     0.8213    0.9827    0.8947       346
DDoS attacks-LOIC-HTTP     0.9999    0.9993    0.9996    115003
 DoS attacks-GoldenEye     0.9996    0.9996    0.9996      8278
      DoS attacks-Hulk     0.9999    1.0000    0.9999

{'model': 'LightGBM_tuned',
 'macro_f1': 0.8566025236714357,
 'accuracy': 0.8730152843543975,
 'balanced_accuracy': np.float64(0.9584224970899368),
 'weighted_f1': 0.9037343613610219,
 'detection_rate': np.float64(0.9836597230884371),
 'false_alarm_rate': np.float64(0.21578531004685694),
 'train_time_s': 1365.133287191391}

**Optuna tuning:** no improvement on the test set (macro-F1 0.8566 vs 0.8574). The validation set used for
tuning had the capped class distribution, not the real one, and the main error source is not a
hyperparameter issue: nearly all false alarms (≈ 71,000 of the 21.6% Benign flows misclassified) are Benign
flows predicted as **Infilteration**, a class whose flows resemble normal traffic and which is known to have
labeling issues in CSE-CIC-IDS2018.

## Summary (fill in after running)
- **Best model and its macro-F1:** `SoftVoting` achieved the highest macro-F1 score of **0.8915** (closely followed by `RandomForest` at **0.8910**).
- **Hardest classes (lowest recall):** **Infilteration** (recall ~0.64 to 0.81 but very low precision, resulting in F1-scores around 0.18 - 0.35 across models) and **Web-Attack** (due to severe class representation issues and resemblance to Benign traffic).
- **Soft voting vs its 3 members:** `SoftVoting` (0.8915) slightly outperformed its individual constituent members: `RandomForest` (0.8910), `XGBoost` (0.8822), and `LightGBM` (0.8574).
- **Effect of Optuna (LightGBM vs LightGBM_tuned):** Optuna tuning did not lead to an improvement on the test set (macro-F1 dropped slightly from **0.8574** to **0.8566**). This is because optimization was performed on a capped distribution, and the main bottleneck is a classification/labeling issue with the **Infilteration** class rather than model hyperparameters.
- **Training time vs performance:**
  - `DecisionTree` was very fast (~35.8 s) with a competitive macro-F1 of **0.8888**.
  - `RandomForest` took ~243 s and achieved **0.8910** macro-F1.
  - Tuning and training `LightGBM_tuned` took ~1365 s but did not yield performance gains over the baseline `RandomForest` or `SoftVoting` ensemble.